In [1]:
import psycopg2
import os
import json
from google.genai import types
from dotenv import load_dotenv
from google import genai
from datetime import date
from pgvector.psycopg2 import register_vector
import mimetypes

In [2]:
# Conexión a la bbdd
load_dotenv("../.env")
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5432"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)
conn.autocommit = True

cur = conn.cursor()
from pgvector.psycopg2 import register_vector
register_vector(conn)
cur.execute("SELECT nombre FROM tipos_ticket ORDER BY nombre")
tipos_permitidos = [r[0] for r in cur.fetchall()]

cur.execute("SELECT nombre FROM categorias_producto ORDER BY nombre")
categorias_permitidas = [r[0] for r in cur.fetchall()]

print(tipos_permitidos)
print(categorias_permitidas)

['cafeteria', 'educacion', 'electronica', 'farmacia', 'gasolinera', 'hogar_decoracion', 'ocio_entretenimiento', 'otro', 'restaurante', 'ropa_moda', 'salud_bienestar', 'seguros', 'suministros_agua', 'suministros_gas', 'suministros_luz', 'supermercado', 'telecomunicaciones', 'transporte']
['alimentacion_general', 'alimentacion_infantil', 'bebidas_alcoholicas', 'bebidas_no_alcoholicas', 'carne', 'cereales_pasta_arroz', 'combustible', 'congelados', 'conservas', 'electronica_tecnologia', 'fruta', 'higiene_personal', 'hogar_menaje', 'lacteos', 'limpieza_hogar', 'mascotas', 'ocio_cultura', 'otros', 'panaderia_bolleria', 'papeleria_oficina', 'pescado_marisco', 'restauracion', 'ropa_calzado', 'salsas_condimentos', 'salud_farmacia', 'snacks_dulces', 'verdura_hortaliza']


In [3]:
prompt = f"""
Analiza esta imagen de un ticket/factura y extrae la información en formato JSON con esta estructura exacta:

{{
  "tipo_ticket": "uno de estos valores: {tipos_permitidos}",
  "comercio": {{
    "nombre": "nombre del establecimiento",
    "direccion": "dirección si aparece, o null",
    "nif": "NIF/CIF si aparece, o null"
  }},
  "fecha": "YYYY-MM-DD",
  "total": 0.00,
  "desglose_iva": [
    {{
      "porcentaje": 0.0,
      "base_imponible": 0.00,
      "cuota": 0.00
    }}
  ],
  "productos": [
    {{
      "descripcion": "texto tal cual aparece en el ticket",
      "categoria": "una de estas: {categorias_permitidas}",
      "cantidad": 0,
      "precio_unitario": 0.00,
      "subtotal": 0.00
    }}
  ]
}}

Reglas:
- "tipo_ticket" y "categoria" deben ser EXACTAMENTE uno de los valores permitidos indicados arriba, sin inventar otros.
- "desglose_iva" debe reflejar la tabla de IVA que aparece en el ticket (normalmente al pie, con columnas de base imponible, porcentaje y cuota). Si el ticket no trae esta tabla, devuelve una lista vacía [].
- Si un producto no encaja claramente en ninguna categoría, usa "alimentacion_general" (para comida) u "otros" (para el resto).
- Si no puedes leer un dato con certeza, pon null en ese campo, no inventes valores.
- Devuelve ÚNICAMENTE el JSON, sin texto adicional ni backticks.
"""

In [4]:
from PIL import Image
from io import BytesIO

def preparar_imagen(ruta_imagen, max_ancho=1600):
    with Image.open(ruta_imagen) as img:
        img = img.convert("RGB")
        if img.width > max_ancho:
            ratio = max_ancho / img.width
            nuevo_alto = int(img.height * ratio)
            img = img.resize((max_ancho, nuevo_alto))
        
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return buffer.getvalue()

In [5]:
ruta_imagen = "../data/samples/PXL_20260731_124417220.jpg"

imagen_bytes = preparar_imagen(ruta_imagen)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        {"inline_data": {"mime_type": "image/jpeg", "data": imagen_bytes}},
        prompt
    ],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_level="low")
    )
)

datos = json.loads(response.text)
print(json.dumps(datos, indent=2, ensure_ascii=False))

{
  "tipo_ticket": "ropa_moda",
  "comercio": {
    "nombre": "H&M",
    "direccion": "C.C Rio Shopping, 47195 Arroyo de la Encomienda",
    "nif": "B82356981"
  },
  "fecha": "2026-06-09",
  "total": 19.99,
  "desglose_iva": [
    {
      "porcentaje": 21.0,
      "base_imponible": 16.52,
      "cuota": 3.47
    }
  ],
  "productos": [
    {
      "descripcion": "Pantalones cortos 1321157 XS Caqui",
      "categoria": "ropa_calzado",
      "cantidad": 1,
      "precio_unitario": 19.99,
      "subtotal": 19.99
    }
  ]
}


In [6]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [7]:
def validar_ticket(datos):
    errores = []
    # Si tipo_tickt no es ninguno de los de la lista
    if datos.get("tipo_ticket") not in tipos_permitidos:
        errores.append(f"tipo_ticket inválido: {datos.get('tipo_ticket')}")

    # Errores en las fechas
    try:
        fecha = date.fromisoformat(datos["fecha"])
        if fecha > date.today():
            errores.append("La fecha está en el futuro")
    except (ValueError, TypeError, KeyError):
        errores.append("Fecha inválida o ausente")

    # Si no hay productos en el ticket
    productos = datos.get("productos", [])
    if not productos:
        errores.append("El ticket no tiene ningún producto")

    
    suma_lineas = 0
    for p in productos:
        # Categorías. Verificamos si las categorias son alguna de las permitidas
        if p.get("categoria") not in categorias_permitidas:
            errores.append(f"Categoría inválida: {p.get('categoria')}")
        
        # Verficamos si el precio es >= 0
        if not p.get("precio_unitario") or p["precio_unitario"] <= 0:
            errores.append(f"Precio no válido en: {p.get('descripcion')}")

        # Verificamos si las cantidades de cada producto son >= 0
        if not p.get("cantidad") or p["cantidad"] <= 0:
            errores.append(f"Cantidad no válida en: {p.get('descripcion')}")
        suma_lineas += (p.get("subtotal") or 0)

    total = datos.get("total") or 0
    # Suma de los precios parciales = suma total del ticket?
    if abs(suma_lineas - total) > 0.05:
        errores.append(f"Suma de líneas ({suma_lineas:.2f}) no coincide con total ({total:.2f})")

    # Verificación extra: si hay desglose de IVA, comprobar que sus bases+cuotas cuadran con el total
    desglose = datos.get("desglose_iva", [])
    if desglose:
        suma_desglose = sum(d["base_imponible"] + d["cuota"] for d in desglose)
        if abs(suma_desglose - total) > 0.05:
            errores.append(f"Desglose de IVA ({suma_desglose:.2f}) no coincide con total ({total:.2f})")

    return errores


errores = validar_ticket(datos)
estado = "validado" if not errores else "pendiente_revision"
motivo_revision = "; ".join(errores) if errores else None

print("Estado:", estado)
print("Errores:", errores if errores else "ninguno")

Estado: validado
Errores: ninguno


In [8]:
# Normalización del comercio
# Se comprueba si existe en la tabla comercios (por C.I.F/N.I.F) y si no existe ese comercio, se le añade.
"""
Busca primero por NIF (si el ticket lo trae) — es el identificador más fiable, porque es único por comercio real, independientemente de cómo esté escrito el nombre.
Si no hay NIF o no encuentra nada, busca por nombre + dirección exactos.
Si no encuentra nada por ninguna vía, crea un registro nuevo.
conn.commit() — esto es importante: sin esta línea, el INSERT quedaría "en el aire", pendiente de confirmar, y no se guardaría de verdad en la base de datos. Es como el botón de "guardar" definitivo de la transacción.
"""
def buscar_o_crear_comercio(comercio_datos, tipo_ticket_nombre):
    cur.execute("SELECT id FROM tipos_ticket WHERE nombre = %s", (tipo_ticket_nombre,))
    tipo_ticket_id = cur.fetchone()[0]

    nif = comercio_datos.get("nif")
    if nif:
        cur.execute("SELECT id FROM comercios WHERE nif = %s", (nif,))
        row = cur.fetchone()
        if row:
            print(f"Comercio encontrado por NIF: id={row[0]}")
            return row[0]

    cur.execute(
        "SELECT id FROM comercios WHERE nombre = %s AND direccion = %s",
        (comercio_datos["nombre"], comercio_datos.get("direccion"))
    )
    row = cur.fetchone()
    if row:
        print(f"Comercio encontrado por nombre+dirección: id={row[0]}")
        return row[0]

    cur.execute(
        """INSERT INTO comercios (nombre, direccion, nif, tipo_ticket_id)
           VALUES (%s, %s, %s, %s) RETURNING id""",
        (comercio_datos["nombre"], comercio_datos.get("direccion"), nif, tipo_ticket_id)
    )
    comercio_id = cur.fetchone()[0]
    conn.commit()
    print(f"Comercio nuevo creado: id={comercio_id}")
    return comercio_id


comercio_id = buscar_o_crear_comercio(datos["comercio"], datos["tipo_ticket"])
print("comercio_id:", comercio_id)

Comercio encontrado por NIF: id=1
comercio_id: 1


In [9]:
def get_embedding(texto, task_type="CLUSTERING"):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=texto,
        config=types.EmbedContentConfig(
            output_dimensionality=1536,
            task_type=task_type
        )
    )
    return result.embeddings[0].values

In [10]:
def buscar_o_crear_producto(descripcion, categoria_nombre, top_k=5):
    cur.execute("SELECT id FROM categorias_producto WHERE nombre = %s", (categoria_nombre,))
    categoria_id = cur.fetchone()[0]

    embedding = get_embedding(descripcion)

    # 1. Embedding: traemos los K candidatos más parecidos de la misma categoría (no decide, solo preselecciona)
    cur.execute(
        """SELECT id, nombre_normalizado
           FROM productos
           WHERE categoria_id = %s
           ORDER BY embedding <=> %s::vector ASC
           LIMIT %s""",
        (categoria_id, embedding, top_k)
    )
    candidatos = cur.fetchall()

    if not candidatos:
        return crear_producto_nuevo(descripcion, categoria_id, embedding)

    # 2. LLM: le mostramos solo esos pocos candidatos y le pedimos que decida
    lista_candidatos = "\n".join([f"{c[0]}: {c[1]}" for c in candidatos])
    prompt_verificacion = f"""
Un ticket menciona el producto: "{descripcion}"

Estos son productos ya existentes en el catálogo, de la misma categoría:
{lista_candidatos}

¿Alguno de ellos es EXACTAMENTE el mismo producto (aunque esté escrito de forma distinta)?
Responde ÚNICAMENTE con el id numérico si hay coincidencia, o con "NINGUNO" si son productos distintos.
"""
    response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[prompt_verificacion],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="minimal")
    )
)
    resultado = response.text.strip()

    if resultado.isdigit():
        print(f"LLM confirma coincidencia: '{descripcion}' -> id={resultado}")
        return int(resultado)

    return crear_producto_nuevo(descripcion, categoria_id, embedding)


def crear_producto_nuevo(descripcion, categoria_id, embedding):
    cur.execute(
        """INSERT INTO productos (nombre_normalizado, categoria_id, embedding)
           VALUES (%s, %s, %s) RETURNING id""",
        (descripcion, categoria_id, embedding)
    )
    producto_id = cur.fetchone()[0]
    print(f"Producto nuevo creado: '{descripcion}' (id={producto_id})")
    return producto_id

In [11]:
for p in datos["productos"]:
    producto_id = buscar_o_crear_producto(p["descripcion"], p["categoria"])
    print(f"  -> producto_id: {producto_id}\n")

LLM confirma coincidencia: 'Pantalones cortos 1321157 XS Caqui' -> id=1
  -> producto_id: 1



In [12]:
def guardar_ticket(datos, ruta_imagen):
    # 1. Validación determinista
    errores = validar_ticket(datos)
    estado = "validado" if not errores else "pendiente_revision"
    motivo_revision = "; ".join(errores) if errores else None

    # 2. Resolver tipo_ticket_id
    cur.execute("SELECT id FROM tipos_ticket WHERE nombre = %s", (datos["tipo_ticket"],))
    tipo_ticket_id = cur.fetchone()[0]

    # 3. Normalizar comercio
    comercio_id = buscar_o_crear_comercio(datos["comercio"], datos["tipo_ticket"])

    # 4. Embedding del ticket completo (para búsquedas tipo "tickets parecidos a este")
    texto_resumen = f"{datos['comercio']['nombre']} - " + ", ".join(p["descripcion"] for p in datos["productos"])
    embedding_ticket = get_embedding(texto_resumen)

    # 5. atributos: guardamos el desglose de IVA tal cual vino
    atributos = json.dumps({"desglose_iva": datos.get("desglose_iva", [])})

    # 6. Insertar la cabecera del ticket
    cur.execute(
        """INSERT INTO tickets (comercio_id, tipo_ticket_id, fecha, total, imagen_path, estado, motivo_revision, atributos, embedding)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s) RETURNING id""",
        (comercio_id, tipo_ticket_id, datos["fecha"], datos["total"], ruta_imagen, estado, motivo_revision, atributos, embedding_ticket)
    )
    ticket_id = cur.fetchone()[0]

    # 7. Insertar cada línea de producto
    for p in datos.get("productos", []):
        producto_id = buscar_o_crear_producto(p["descripcion"], p["categoria"])
        cur.execute(
            """INSERT INTO lineas_ticket (ticket_id, producto_id, descripcion_original, cantidad, precio_unitario, subtotal)
               VALUES (%s, %s, %s, %s, %s, %s)""",
            (ticket_id, producto_id, p["descripcion"], p["cantidad"], p["precio_unitario"], p["subtotal"])
        )

    print(f"\nTicket {ticket_id} guardado. Estado: {estado}")
    if motivo_revision:
        print(f"Motivo: {motivo_revision}")

    return ticket_id

In [13]:
ticket_id = guardar_ticket(datos, ruta_imagen)

Comercio encontrado por NIF: id=1
LLM confirma coincidencia: 'Pantalones cortos 1321157 XS Caqui' -> id=1

Ticket 5 guardado. Estado: validado


In [14]:
cur.close()
conn.close()

In [15]:
import sys
sys.path.append("..")

from agents.extraction.config import get_gemini_client, get_db_connection
from agents.extraction.pipeline import procesar_ticket
import json

client = get_gemini_client()
conn = get_db_connection()
cur = conn.cursor()

resultado = procesar_ticket(cur, client, "../data/samples/PXL_20260322_160648157.jpg")
print(json.dumps(resultado, indent=2, ensure_ascii=False, default=str))

{
  "ticket_id": 6,
  "estado": "pendiente_revision",
  "motivo_revision": "Precio no válido en: PIMIENTO ROJO CAT. 1ª; Cantidad no válida en: PIMIENTO ROJO CAT. 1ª; Precio no válido en: PIMIENTO DE FREIR CAT. 1ª; Cantidad no válida en: PIMIENTO DE FREIR CAT. 1ª; Precio no válido en: CEBOLLA GRANEL; Cantidad no válida en: CEBOLLA GRANEL; Precio no válido en: MEZCLA FR.ROJOS LA CUERVA 300G; Cantidad no válida en: MEZCLA FR.ROJOS LA CUERVA 300G; Precio no válido en: CAFE VIVO SOLUBLE NAT.200 GR; Cantidad no válida en: CAFE VIVO SOLUBLE NAT.200 GR; Precio no válido en: HUEVO COMAVI L 63/73 DOC.; Cantidad no válida en: HUEVO COMAVI L 63/73 DOC.",
  "datos_extraidos": {
    "tipo_ticket": "supermercado",
    "comercio": {
      "nombre": "Tifer",
      "direccion": "C/Los Trigales, 19. Eras del Bosque, Palencia",
      "nif": "A39050349"
    },
    "fecha": "2026-03-16",
    "total": 11.76,
    "desglose_iva": [
      {
        "porcentaje": 4.0,
        "base_imponible": 7.24,
        "cuo